# Project: End-to-End RAG Application

## Overview
This project integrates everything from the three lab tracks into a single, production-ready pipeline:

| Lab Track | What You Built | Used Here |
|-----------|---------------|----------|
| **Tokenization & API** | Token counting, cost estimation, LiteLLM client | Token budget guard, LLM generation |
| **Ingestion & Embedding** | `Document` model, chunkers, `EmbeddingGenerator` | Full ingestion pipeline |
| **RAG Service** | `VectorStoreManager`, `RAGService` | Retrieval + answer generation |
| **Observability** | `AgentTracer`, `@observe`, DeepEval | End-to-end tracing + evaluation |
| **Multi-Agent** | Planner, specialist agents | Query routing & rewriting |

## What You Will Build
A **Research Assistant** that:
1. Ingests documents (web pages + text files) through a dispatcher
2. Chunks and embeds them into ChromaDB
3. Answers queries using a traced RAG pipeline with score thresholding
4. Routes complex queries through a multi-agent planner
5. Evaluates answer quality with DeepEval

## Deliverables
Complete every `# TODO` cell. The `checks` module will validate each step.

---
## Phase 0: Setup

In [ ]:
!uv pip install litellm python-dotenv chromadb numpy langchain-text-splitters \
               sentence-transformers requests beautifulsoup4 deepeval tiktoken -q

In [ ]:
import os
import re
import time
import json
import uuid
import hashlib
import logging
import unicodedata
import numpy as np
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
from abc import ABC, abstractmethod

import requests
from bs4 import BeautifulSoup
import chromadb
import tiktoken

import logging
logging.getLogger("deepeval").setLevel(logging.ERROR)
os.environ["DEEPEVAL_TELEMETRY"] = "0"
os.environ["CONFIDENT_AI_API_KEY"] = "dummy"

from dotenv import load_dotenv
load_dotenv()

MODEL_ID = os.getenv('MODEL_NAME', 'openrouter/nvidia/nemotron-3-super-120b-a12b:free')

if not os.getenv('OPENROUTER_API_KEY'):
    print('WARNING: OPENROUTER_API_KEY not found. Set it in your .env file.')
else:
    print(f'Environment loaded. Model: {MODEL_ID}')

---
## Phase 1: Document Ingestion

Re-use the `Document` model and pipeline from Lab 1 (Ingestion track). This is your single contract for all downstream components.

In [ ]:
@dataclass
class Document:
    """Standardized representation of an ingested document."""
    content: str
    source: str
    title: Optional[str] = None
    doc_type: str = "unknown"
    author: Optional[str] = None
    ingested_at: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    word_count: int = 0
    extra_metadata: Dict = field(default_factory=dict)

    def __post_init__(self):
        if self.content and self.word_count == 0:
            self.word_count = len(self.content.split())

    def to_dict(self) -> dict:
        return {
            "content": self.content,
            "source": self.source,
            "metadata": {
                "title": self.title,
                "type": self.doc_type,
                "author": self.author,
                "word_count": self.word_count,
            }
        }

print('Document model ready.')

In [ ]:
def clean_text(text: str) -> str:
    """Master cleaning function for extracted text."""
    if not text:
        return ""
    text = text.replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"Page \d+ of \d+", "", text)
    text = re.sub(r"-\n(\w)", r"\1", text)
    return text.strip()


def extract_web_page(url: str) -> Document:
    """Extract clean text content from a web page."""
    resp = requests.get(url, headers={"User-Agent": "ResearchAssistant/1.0"}, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header", "form"]):
        tag.decompose()
    content = soup.find("main") or soup.find("article") or soup.body
    text = content.get_text(separator="\n", strip=True) if content else ""
    title = soup.title.string if soup.title else url
    return Document(content=clean_text(text), source=url, title=title, doc_type="web")


def extract_markdown(file_path: str) -> Document:
    """Extract content from a markdown file with basic metadata."""
    text = Path(file_path).read_text(encoding="utf-8")
    title = None
    for line in text.splitlines():
        if line.startswith("# "):
            title = line[2:].strip()
            break
    return Document(content=clean_text(text), source=file_path, title=title, doc_type="markdown")


def extract_document(source: str) -> Document:
    """Route to the correct extractor based on source type."""
    if source.startswith(("http://", "https://")):
        return extract_web_page(source)
    ext = Path(source).suffix.lower()
    if ext == ".md":
        return extract_markdown(source)
    elif ext == ".txt":
        text = Path(source).read_text(encoding="utf-8")
        return Document(content=clean_text(text), source=source, doc_type="text")
    else:
        raise ValueError(f"Unsupported format: {ext}")

print('Ingestion pipeline ready.')

---
## Phase 2: Chunking & Embedding

Re-use `BaseChunker`, `RecursiveChunker`, and `EmbeddingGenerator` from Lab 2 (Embedding track).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

class BaseChunker(ABC):
    """Abstract base class for all chunking strategies."""
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    @abstractmethod
    def chunk_document(self, text: str, metadata: dict = None) -> List[Dict[str, Any]]:
        pass

    def _create_chunk_dict(self, text: str, metadata: dict, chunk_id: int) -> dict:
        chunk_meta = metadata.copy() if metadata else {}
        chunk_meta.update({
            'chunk_id': chunk_id,
            'char_length': len(text),
            'chunker': self.__class__.__name__
        })
        return {'text': text.strip(), 'metadata': chunk_meta}


class RecursiveChunker(BaseChunker):
    """Chunks respecting paragraph and sentence boundaries."""
    def __init__(self, chunk_size=500, chunk_overlap=50, separators=None):
        super().__init__(chunk_size, chunk_overlap)
        self.separators = separators or ["\n\n", "\n", ". ", " ", ""]
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            separators=self.separators,
        )

    def chunk_document(self, text, metadata=None):
        text_chunks = self._splitter.split_text(text)
        return [self._create_chunk_dict(t, metadata, i) for i, t in enumerate(text_chunks)]


class EmbeddingGenerator:
    """Simulates embedding generation with batching and retry logic."""
    def __init__(self, model_name="text-embedding-3-small", dimension=384):
        self.model_name = model_name
        self.dimension = dimension
        self.total_tokens = 0

    def _embed_batch(self, texts: list, attempt=1, max_retries=3) -> list:
        """Generate embeddings for a batch with retry logic."""
        try:
            time.sleep(0.01 * len(texts))
            # In production: replace with real OpenAI API call
            embeddings = [np.random.randn(self.dimension).tolist() for _ in texts]
            self.total_tokens += sum(len(t.split()) * 1.3 for t in texts)
            return embeddings
        except Exception as e:
            if attempt < max_retries:
                wait = 2 ** attempt
                print(f"  Retry {attempt}/{max_retries} after {wait}s: {e}")
                time.sleep(wait * 0.01)
                return self._embed_batch(texts, attempt + 1, max_retries)
            raise

    def generate_embeddings(self, texts: list, batch_size=100) -> list:
        """Process all texts in batches."""
        all_embeddings = []
        total_batches = (len(texts) + batch_size - 1) // batch_size
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            batch_num = i // batch_size + 1
            print(f"  Batch {batch_num}/{total_batches} ({len(batch)} texts)...")
            all_embeddings.extend(self._embed_batch(batch))
        print(f"Generated {len(all_embeddings)} embeddings, ~{self.total_tokens:.0f} tokens used")
        return all_embeddings

print('Chunking & embedding components ready.')

---
## Phase 3: Vector Store

Re-use `VectorStoreManager` from Lab 3 (RAG Service track).

In [ ]:
class VectorStoreManager:
    """Manages ChromaDB with HNSW indexing."""

    def __init__(self, collection_name="research_papers"):
        self.client = chromadb.Client()
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={
                "hnsw:space": "cosine",
                "hnsw:construction_ef": 200,
                "hnsw:search_ef": 100,
                "hnsw:M": 16,
            }
        )
        print(f"Collection '{collection_name}' ready (HNSW cosine)")

    def add_documents(self, documents, batch_size=100):
        """Add documents with embeddings in batches."""
        total = 0
        for i in range(0, len(documents), batch_size):
            batch = documents[i:i + batch_size]
            ids, embeddings, metadatas, texts = [], [], [], []
            for doc in batch:
                content_hash = hashlib.md5(doc['text'].encode()).hexdigest()
                ids.append(f"doc_{content_hash}")
                embeddings.append(doc['embedding'])
                # Ensure all metadata values are strings (ChromaDB requirement)
                safe_meta = {k: str(v) for k, v in doc.get('metadata', {}).items()}
                metadatas.append(safe_meta)
                texts.append(doc['text'])
            self.collection.upsert(
                embeddings=embeddings,
                documents=texts,
                metadatas=metadatas,
                ids=ids
            )
            total += len(batch)
        print(f"Upserted {total} documents. Collection size: {self.collection.count()}")
        return total

    def search(self, query_embedding, n_results=5, filter_conditions=None):
        """Search for similar documents."""
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where=filter_conditions,
            include=["documents", "metadatas", "distances"]
        )
        formatted = []
        for i in range(len(results["documents"][0])):
            formatted.append({
                "text": results["documents"][0][i],
                "metadata": results["metadatas"][0][i],
                "score": 1 - results["distances"][0][i]
            })
        return formatted

    def get_stats(self):
        return {"count": self.collection.count()}

print('VectorStoreManager ready.')

---
## Phase 4: Observability

Re-use `AgentTracer` from Lab 1 (Observability track). Every pipeline call is wrapped in a trace.

In [ ]:
@dataclass
class ToolCallRecord:
    tool_name: str
    tool_input: dict
    tool_output: str
    duration_ms: float

@dataclass
class AgentStep:
    step_number: int
    reasoning: Optional[str] = None
    tool_calls: list = field(default_factory=list)
    cost_usd: float = 0.0

@dataclass
class Trace:
    trace_id: str
    agent_name: str
    steps: list = field(default_factory=list)
    status: str = 'running'
    total_cost_usd: float = 0.0

class AgentTracer:
    def __init__(self):
        self.traces = {}

    def start_trace(self, trace_id, agent_name):
        self.traces[trace_id] = Trace(trace_id=trace_id, agent_name=agent_name)
        return trace_id

    def log_step(self, trace_id, step):
        if trace_id in self.traces:
            self.traces[trace_id].steps.append(step)
            self.traces[trace_id].total_cost_usd += step.cost_usd

    def end_trace(self, trace_id, status='success'):
        if trace_id in self.traces:
            self.traces[trace_id].status = status

    def print_trace(self, trace_id):
        t = self.traces.get(trace_id)
        if not t:
            return
        print(f'\n=== TRACE [{t.trace_id}] Agent: {t.agent_name} | Status: {t.status} ===')
        for step in t.steps:
            print(f'  Step {step.step_number}: cost=${step.cost_usd:.4f}')
            for tc in step.tool_calls:
                print(f'    -> {tc.tool_name}({tc.tool_input}) [{tc.duration_ms:.0f}ms] => {str(tc.tool_output)[:80]}')
        print(f'  Total cost: ${t.total_cost_usd:.4f}')

tracer = AgentTracer()
print('AgentTracer ready.')

---
## Phase 5: Token Budget Guard

From Lab 1 (Tokenization track): before sending a prompt to the LLM, verify it fits within the context window and estimate the cost.

In [ ]:
PRICING_PER_1M = {
    "gpt-4o":              {"input": 2.50,  "output": 10.00},
    "gpt-3.5-turbo":       {"input": 0.50,  "output": 1.50},
    "claude-3-5-sonnet":   {"input": 3.00,  "output": 15.00},
    "default":             {"input": 1.00,  "output": 3.00},
}

def count_tokens(text: str, model: str = "gpt-4") -> int:
    """Count tokens for a given text."""
    try:
        enc = tiktoken.encoding_for_model(model)
    except Exception:
        enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))


def check_token_budget(prompt: str, max_tokens: int = 8192, model: str = "gpt-4") -> dict:
    """Verify prompt fits within budget and estimate cost."""
    token_count = count_tokens(prompt, model)
    fits = token_count <= max_tokens
    prices = PRICING_PER_1M.get("default")
    estimated_cost = (token_count / 1_000_000) * prices["input"]
    return {
        "token_count": token_count,
        "max_tokens": max_tokens,
        "fits": fits,
        "utilization_pct": round(token_count / max_tokens * 100, 1),
        "estimated_input_cost_usd": estimated_cost
    }

# Quick test
budget = check_token_budget("Hello, world! This is a test prompt.")
print(f"Token budget check: {budget}")
print('Token budget guard ready.')

---
## Phase 6: The Traced RAG Service

Combines the `RAGService` from Lab 3 with the `AgentTracer` from Lab 1. Every call is fully observable.

In [ ]:
import litellm

DIM = 384

class TracedRAGService:
    """Orchestrates Retrieve -> Augment -> Generate with full observability."""

    def __init__(self, vector_store: VectorStoreManager, tracer: AgentTracer,
                 model_id: str = MODEL_ID, embed_dim: int = DIM):
        self.vector_store = vector_store
        self.tracer = tracer
        self.model_id = model_id
        self.embed_dim = embed_dim
        self.logger = logging.getLogger('RAGService')
        logging.basicConfig(level=logging.INFO,
                            format='%(asctime)s %(levelname)s %(message)s',
                            datefmt='%H:%M:%S')

    def _embed_query(self, query: str) -> list:
        """Generate query embedding. In production: call OpenAI API."""
        emb = np.random.randn(self.embed_dim) * 0.1
        q = query.lower()
        if any(w in q for w in ["rag", "retrieval", "generation", "augmented"]):
            emb[:50] += 0.5
        if any(w in q for w in ["vector", "embedding", "cosine", "similarity"]):
            emb[50:100] += 0.5
        if any(w in q for w in ["chunk", "split", "segmentation"]):
            emb[100:150] += 0.5
        if any(w in q for w in ["transformer", "attention", "self-attention"]):
            emb[150:200] += 0.5
        return (emb / np.linalg.norm(emb)).tolist()

    def _build_context(self, results: list) -> str:
        """Format retrieved chunks into context string."""
        parts = []
        for i, doc in enumerate(results, 1):
            title = doc['metadata'].get('title', 'Unknown')
            score = doc['score']
            parts.append(f"[{i}] (Source: {title} | Score: {score:.3f})\n{doc['text']}")
        return "\n\n---\n\n".join(parts)

    def _build_prompt(self, query: str, context: str) -> str:
        """Build the full grounded prompt for the LLM."""
        system = """You are a Research Assistant. Answer using ONLY the provided context.
Cite sources using [N] format. If the answer is not in the context, say \"I don't have enough information.\""""
        return f"""SYSTEM: {system}

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

    def answer(self, query: str, top_k: int = 3, min_score: float = 0.0) -> dict:
        """Full traced RAG pipeline: Retrieve -> Augment -> Generate."""
        trace_id = str(uuid.uuid4())[:8]
        self.tracer.start_trace(trace_id, 'TracedRAGService')

        # STEP 1 — RETRIEVE
        t0 = time.time()
        query_embedding = self._embed_query(query)
        results = self.vector_store.search(query_embedding, n_results=top_k)
        results = [r for r in results if r['score'] >= min_score]
        retrieve_ms = (time.time() - t0) * 1000

        step1 = AgentStep(step_number=1, reasoning=f"Retrieved {len(results)} chunks above score {min_score}")
        step1.tool_calls.append(ToolCallRecord(
            tool_name='vector_search',
            tool_input={'query': query[:60], 'top_k': top_k, 'min_score': min_score},
            tool_output=f"{len(results)} results",
            duration_ms=retrieve_ms
        ))
        self.tracer.log_step(trace_id, step1)

        if not results:
            self.tracer.end_trace(trace_id, status='no_results')
            return {"answer": "No relevant information found above the score threshold.",
                    "sources": [], "trace_id": trace_id}

        # STEP 2 — AUGMENT
        context = self._build_context(results)
        prompt = self._build_prompt(query, context)

        budget = check_token_budget(prompt)
        self.logger.info(f"Prompt: {budget['token_count']} tokens ({budget['utilization_pct']}% of budget)")

        # STEP 3 — GENERATE
        t0 = time.time()
        try:
            response = litellm.completion(
                model=self.model_id,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=512,
                temperature=0.1,
                num_retries=3,
                timeout=60,
            )
            answer_text = response.choices[0].message.content
            cost = getattr(response, '_hidden_params', {}).get('response_cost', 0.0) or 0.0
        except Exception as e:
            answer_text = f"[Generation error: {e}]"
            cost = 0.0
        generate_ms = (time.time() - t0) * 1000

        step3 = AgentStep(step_number=3, reasoning=answer_text[:120], cost_usd=cost)
        step3.tool_calls.append(ToolCallRecord(
            tool_name='llm_generate',
            tool_input={'model': self.model_id, 'prompt_tokens': budget['token_count']},
            tool_output=answer_text[:80],
            duration_ms=generate_ms
        ))
        self.tracer.log_step(trace_id, step3)
        self.tracer.end_trace(trace_id, status='success')

        return {
            "answer": answer_text,
            "sources": [r['metadata'] for r in results],
            "scores": [r['score'] for r in results],
            "trace_id": trace_id,
            "budget": budget
        }

print('TracedRAGService ready.')

---
## Phase 7: Multi-Agent Query Router

From Lab 2 (Multi-Agent track): for complex queries, a **Planner** decomposes the task and routes sub-tasks to specialist agents before the RAG lookup.

In [ ]:
QUERY_COMPLEXITY_PROMPT = """You are a query classifier for a research assistant.
Classify the query as one of:
- SIMPLE   : a single factual lookup (e.g. "What is RAG?")
- COMPLEX  : requires comparison, synthesis, or multi-step reasoning
- AMBIGUOUS: unclear intent, needs rewriting

Respond with ONLY the label and a one-line reason.
Format: <LABEL>: <reason>

Query: {query}"""

QUERY_REWRITE_PROMPT = """You are a query rewriting specialist.
Rewrite the ambiguous query into a clear, specific research question.
Return ONLY the rewritten query, nothing else.

Original query: {query}"""


class QueryRouter:
    """Routes queries: simple -> RAG direct, complex -> planner, ambiguous -> rewrite."""

    def __init__(self, rag_service: TracedRAGService, model_id: str = MODEL_ID):
        self.rag = rag_service
        self.model_id = model_id
        self.logger = logging.getLogger('QueryRouter')

    def _call_llm(self, prompt: str, max_tokens: int = 100) -> str:
        """Single LLM call helper."""
        try:
            resp = litellm.completion(
                model=self.model_id,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=0,
                num_retries=3,
                timeout=30,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            return f"ERROR: {e}"

    def classify(self, query: str) -> tuple[str, str]:
        """Returns (label, reason)."""
        raw = self._call_llm(QUERY_COMPLEXITY_PROMPT.format(query=query))
        parts = raw.split(':', 1)
        label = parts[0].strip().upper() if parts else 'SIMPLE'
        reason = parts[1].strip() if len(parts) > 1 else raw
        if label not in ('SIMPLE', 'COMPLEX', 'AMBIGUOUS'):
            label = 'SIMPLE'
        return label, reason

    def rewrite(self, query: str) -> str:
        """Rewrite an ambiguous query."""
        return self._call_llm(QUERY_REWRITE_PROMPT.format(query=query), max_tokens=80)

    def route(self, query: str, top_k: int = 3, min_score: float = 0.0) -> dict:
        """Classify, optionally rewrite, then answer."""
        self.logger.info(f'Routing: "{query[:60]}"')
        label, reason = self.classify(query)
        self.logger.info(f'Classification: {label} — {reason}')

        if label == 'AMBIGUOUS':
            rewritten = self.rewrite(query)
            self.logger.info(f'Rewritten to: "{rewritten}"')
            query = rewritten

        result = self.rag.answer(query, top_k=top_k, min_score=min_score)
        result['route_label'] = label
        result['route_reason'] = reason
        return result

print('QueryRouter ready.')

---
## Phase 8: Wire Everything Together

Ingest sample documents, populate the vector store, and build the full application.

In [ ]:
np.random.seed(42)

# --- Sample knowledge base (simulates ingested documents) ---
SAMPLE_DOCS = [
    {"text": "RAG (Retrieval-Augmented Generation) combines a retrieval system with a generative model. The retriever fetches relevant documents from a knowledge base; the generator produces an answer grounded in those documents. This reduces hallucinations and allows citing sources.",
     "metadata": {"title": "RAG Overview", "section": "introduction", "doc_type": "web"}},
    {"text": "The transformer architecture processes all sequence positions in parallel using self-attention. Each token attends to every other token, producing context-aware representations. Positional encodings are added to token embeddings to preserve order information.",
     "metadata": {"title": "Attention Is All You Need", "section": "architecture", "doc_type": "pdf"}},
    {"text": "HNSW (Hierarchical Navigable Small World) is the standard approximate nearest-neighbor algorithm for vector databases. It builds a multi-layered proximity graph, enabling sub-linear search time with high recall. The hnsw:M parameter controls graph connectivity.",
     "metadata": {"title": "HNSW Algorithm", "section": "algorithm", "doc_type": "pdf"}},
    {"text": "Cosine similarity measures the angle between two embedding vectors. It is length-invariant, making it ideal for text embeddings where document length should not bias similarity. Values range from -1 (opposite) to 1 (identical direction).",
     "metadata": {"title": "Vector Similarity Guide", "section": "metrics", "doc_type": "web"}},
    {"text": "RAG vs Fine-tuning: RAG keeps the base model frozen and retrieves knowledge dynamically, making it cheaper to update and audit. Fine-tuning bakes knowledge into weights, giving faster inference but requiring retraining when knowledge changes.",
     "metadata": {"title": "RAG vs Fine-tuning", "section": "comparison", "doc_type": "web"}},
    {"text": "Chunking splits long documents into smaller pieces before embedding. Recursive chunking respects paragraph and sentence boundaries, producing semantically coherent chunks. Overlap (typically 10-15% of chunk size) prevents context loss at boundaries.",
     "metadata": {"title": "Chunking Strategies", "section": "fundamentals", "doc_type": "web"}},
    {"text": "BM25 is a probabilistic keyword ranking function. It scores documents based on term frequency and inverse document frequency. In hybrid RAG systems, BM25 scores are combined with cosine similarity scores using Reciprocal Rank Fusion (RRF).",
     "metadata": {"title": "Hybrid Search", "section": "bm25", "doc_type": "pdf"}},
    {"text": "DeepEval provides automated evaluation of RAG systems using LLM-as-Judge. Key metrics include: Faithfulness (answer grounded in context?), Answer Relevancy (answers the question?), and Contextual Recall (context covers the answer?).",
     "metadata": {"title": "RAG Evaluation", "section": "metrics", "doc_type": "web"}},
    {"text": "Token costs vary dramatically across models. GPT-4-Turbo charges $10/1M input tokens while GPT-3.5-Turbo charges $0.50/1M. For a 10-page document (~7,500 tokens), GPT-4-Turbo costs ~$0.075 vs $0.004 for GPT-3.5-Turbo.",
     "metadata": {"title": "Token Cost Analysis", "section": "economics", "doc_type": "web"}},
    {"text": "Observability in agentic systems requires tracing each step: reasoning, tool calls, inputs, outputs, latency, and cost. Tools like Langfuse and DeepEval enable production monitoring. Loop detection prevents agents from repeating the same failing tool call indefinitely.",
     "metadata": {"title": "Agent Observability", "section": "monitoring", "doc_type": "web"}},
]

# Generate topic-aware embeddings
for chunk in SAMPLE_DOCS:
    base = np.random.randn(DIM) * 0.1
    t = chunk['text'].lower()
    if any(w in t for w in ['rag', 'retrieval', 'augmented']):
        base[:50] += 0.6
    if any(w in t for w in ['vector', 'embedding', 'cosine', 'hnsw']):
        base[50:100] += 0.6
    if any(w in t for w in ['chunk', 'split', 'overlap']):
        base[100:150] += 0.6
    if any(w in t for w in ['transformer', 'attention', 'self-attention']):
        base[150:200] += 0.6
    if any(w in t for w in ['cost', 'token', 'pricing']):
        base[200:250] += 0.6
    chunk['embedding'] = (base / np.linalg.norm(base)).tolist()

# Build the application
store = VectorStoreManager(collection_name="project_kb")
store.add_documents(SAMPLE_DOCS)

rag_service = TracedRAGService(vector_store=store, tracer=tracer, model_id=MODEL_ID)
router = QueryRouter(rag_service=rag_service, model_id=MODEL_ID)

print(f'\nApplication ready. Knowledge base: {store.get_stats()["count"]} chunks.')

---
## Phase 9: Run Queries

Test the full pipeline end-to-end. Change the queries below and observe the traces.

In [ ]:
# Direct RAG (no routing) — fast path for simple known queries
result = rag_service.answer(
    query="What is RAG and how does it reduce hallucinations?",
    top_k=3,
    min_score=0.0
)

print(f"Answer:\n{result['answer']}")
print(f"\nSources:")
for s in result['sources']:
    print(f"  - {s.get('title', 'N/A')} [{s.get('section', 'N/A')}]")
print(f"\nRetrieval scores: {[f'{s:.3f}' for s in result['scores']]}")
print(f"Token budget: {result['budget']}")

# Print the trace
tracer.print_trace(result['trace_id'])

In [ ]:
# Routed query — the QueryRouter classifies and optionally rewrites first
result2 = router.route(
    query="Compare RAG with fine-tuning — when should I use each?",
    top_k=3,
    min_score=0.0
)

print(f"Route label: {result2.get('route_label')} — {result2.get('route_reason')}")
print(f"\nAnswer:\n{result2['answer']}")
tracer.print_trace(result2['trace_id'])

---
## Phase 10: Automated Evaluation

From Lab 1 (Observability track): use **DeepEval** to score the pipeline's answers programmatically.

In [ ]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import TaskCompletionMetric
from deepeval.models import LiteLLMModel

judge_model = LiteLLMModel(model=MODEL_ID, temperature=0)

# Collect test cases from the queries above
test_cases = [
    LLMTestCase(
        input="What is RAG and how does it reduce hallucinations?",
        actual_output=result['answer'],
        expected_output="RAG retrieves relevant documents and grounds the LLM answer in that context, reducing hallucinations by providing factual sources."
    ),
    LLMTestCase(
        input="Compare RAG with fine-tuning.",
        actual_output=result2['answer'],
        expected_output="RAG keeps the model frozen and retrieves knowledge dynamically; fine-tuning bakes knowledge into weights but requires retraining when knowledge changes."
    ),
]

task_metric = TaskCompletionMetric(threshold=0.5, model=judge_model)

print('Running DeepEval evaluation...')
eval_results = evaluate(test_cases=test_cases, metrics=[task_metric])
print('Evaluation complete.')

---
## Exercises

Complete each `# TODO` below. The `checks` module will validate your implementation.

### Exercise 1: Ingestion — Add a Text File Source

The `extract_document` dispatcher already handles URLs and `.md` files. Add a document from a plain text string (simulating a `.txt` file) and verify it is chunked and stored correctly.

In [ ]:
import tempfile

sample_txt_content = """Hybrid search combines vector similarity search with BM25 keyword search.
Results from both retrievers are merged using Reciprocal Rank Fusion (RRF).
This approach outperforms either method alone, especially for queries with rare terms.
Production RAG systems at Elasticsearch, Weaviate, and Qdrant all support hybrid search natively.
"""

# TODO: Write sample_txt_content to a temporary .txt file, extract it using
# extract_document(), chunk it with RecursiveChunker(chunk_size=300, chunk_overlap=30),
# generate embeddings, and add to the store.

with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8') as f:
    f.write(sample_txt_content)
    temp_txt_path = f.name

try:
    # TODO 1: Extract the document
    new_doc = None  # Replace with: extract_document(temp_txt_path)

    # TODO 2: Chunk the document
    chunker = None  # Replace with: RecursiveChunker(chunk_size=300, chunk_overlap=30)
    chunks = []     # Replace with: chunker.chunk_document(new_doc.content, {"title": new_doc.title or "txt_doc", "doc_type": "text"})

    # TODO 3: Generate embeddings and attach them to each chunk
    generator = EmbeddingGenerator(dimension=DIM)
    texts = [c['text'] for c in chunks]
    embeddings = []  # Replace with: generator.generate_embeddings(texts)
    for chunk, emb in zip(chunks, embeddings):
        chunk['embedding'] = emb

    # TODO 4: Add chunks to the vector store
    # store.add_documents(chunks)

    from tests import checks
    checks.check_project_ex1(new_doc, chunks, store)
finally:
    os.unlink(temp_txt_path)

### Exercise 2: Retrieval — Implement Metadata Filtering

Search the vector store for chunks where `doc_type == "pdf"` only. This simulates a user who wants answers sourced exclusively from research papers.

In [ ]:
# TODO: Create a query embedding for "transformer self-attention mechanism"
# and call store.search() with a filter_conditions dict that restricts results
# to doc_type == "pdf". Print the results.

query_text = "transformer self-attention mechanism"

# TODO 1: Build a query embedding
query_emb = None  # Replace with: rag_service._embed_query(query_text)

# TODO 2: Search with metadata filter
pdf_results = None  # Replace with: store.search(query_emb, n_results=5, filter_conditions={"doc_type": "pdf"})

assert pdf_results is not None, "Call store.search with filter_conditions"
assert len(pdf_results) > 0, "Should find at least one PDF chunk"
assert all(r['metadata']['doc_type'] == 'pdf' for r in pdf_results), \
    "All results must have doc_type == pdf"

print(f"Found {len(pdf_results)} PDF results for '{query_text}':")
for r in pdf_results:
    print(f"  [{r['score']:.3f}] {r['metadata']['title']}: {r['text'][:80]}...")

from tests import checks
checks.check_project_ex2(pdf_results)

### Exercise 3: Observability — Add Loop Detection to the RAG Pipeline

Extend `TracedRAGService.answer()` with a `LoopDetector` that raises a warning if the same query is answered more than `threshold` times in the same session. This prevents runaway agent loops.

In [ ]:
class LoopDetector:
    def __init__(self, threshold=2):
        self.history = []
        self.threshold = threshold

    def check(self, name: str, args: str) -> bool:
        """Returns True if (name, args) has been seen >= threshold times."""
        call = (name, args)
        count = self.history.count(call)
        self.history.append(call)
        return count >= self.threshold


class LoopAwareRAGService(TracedRAGService):
    """TracedRAGService with loop detection."""

    def __init__(self, *args, loop_threshold=2, **kwargs):
        super().__init__(*args, **kwargs)
        # TODO: Initialize a LoopDetector with loop_threshold
        self.loop_detector = None  # Replace with: LoopDetector(threshold=loop_threshold)

    def answer(self, query: str, top_k: int = 3, min_score: float = 0.0) -> dict:
        # TODO: Before calling super().answer(), check if this query has been
        # seen too many times using self.loop_detector.check('rag_answer', query).
        # If a loop is detected, return a dict with answer='Loop detected: query repeated too many times.'
        pass  # Replace with your implementation

# Test loop detection
loop_rag = LoopAwareRAGService(vector_store=store, tracer=tracer, model_id=MODEL_ID, loop_threshold=2)

repeated_query = "What is cosine similarity?"
for i in range(3):
    r = loop_rag.answer(repeated_query)
    print(f"Call {i+1}: {r['answer'][:80]}")

from tests import checks
checks.check_project_ex3(LoopDetector, LoopAwareRAGService, store, tracer, MODEL_ID)

### Exercise 4: Cost Estimation — Pre-flight Budget Check

Before ingesting a large corpus, estimate the total embedding cost. Implement `plan_ingestion_cost()` that reports token counts and cost for a list of documents.

In [ ]:
def plan_ingestion_cost(
    documents: List[Document],
    chunk_size: int = 500,
    chunk_overlap: int = 50,
    embedding_model: str = "text-embedding-3-small"
) -> dict:
    """
    Estimate the cost of chunking and embedding a list of documents.

    Returns a dict with:
      - total_documents: int
      - total_chunks: int
      - total_tokens: int
      - cost_usd: float
      - model: str
    """
    EMBED_PRICING = {
        "text-embedding-3-small": 0.02,
        "text-embedding-3-large": 0.13,
    }
    enc = tiktoken.get_encoding("cl100k_base")

    # TODO: For each document, chunk it with RecursiveChunker and count tokens
    # Sum up total_chunks and total_tokens across all documents.
    # Calculate cost_usd = (total_tokens / 1_000_000) * EMBED_PRICING[embedding_model]

    total_chunks = 0
    total_tokens = 0

    # TODO: Implement the loop here

    cost_usd = None  # Replace with: (total_tokens / 1_000_000) * EMBED_PRICING[embedding_model]

    return {
        "total_documents": len(documents),
        "total_chunks": total_chunks,
        "total_tokens": total_tokens,
        "cost_usd": cost_usd,
        "model": embedding_model
    }

# Build test documents from sample chunks
test_docs = [
    Document(content=c['text'], source="sample", title=c['metadata']['title'])
    for c in SAMPLE_DOCS
]

plan = plan_ingestion_cost(test_docs)
print(f"Ingestion plan:")
for k, v in plan.items():
    print(f"  {k}: {v}")

from tests import checks
checks.check_project_ex4(plan)

---
## Reflection Questions

Answer each question in the cell below.

*Your answers here:*

1. **Chunking strategy**: The project uses `RecursiveChunker`. For a corpus of Jupyter notebooks (`.ipynb` files), which separator hierarchy would you change and why?

   *...*

2. **Score thresholding**: If you raise `min_score` from 0.0 to 0.7, what risks do you introduce? How would you choose the right threshold for a production system?

   *...*

3. **Observability gap**: The `TracedRAGService` traces retrieval and generation. What important event is NOT traced yet, and how would you add it?

   *...*

4. **Cost at scale**: You need to ingest 50,000 Wikipedia articles averaging 1,200 words each. Estimate the total embedding cost using `text-embedding-3-small`. Show your calculation.

   *...*